In [48]:
import cryo
import polars as pl
import binascii
import web3
import json
from eth_abi import decode
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import pandas as pd

In [49]:
# The multical cantract address, but we also need ABI
MULTICALL3_ADDRESS = '0xcA11bde05977b3631167028862bE2a173976CA11'
MULTICALL3_ABI=json.loads('[{"inputs":[{"internalType":"bool","name":"requireSuccess","type":"bool"},{"components":[{"internalType":"address","name":"target","type":"address"},{"internalType":"bytes","name":"callData","type":"bytes"}],"internalType":"struct Multicall3.Call[]","name":"calls","type":"tuple[]"}],"name":"tryAggregate","outputs":[{"components":[{"internalType":"bool","name":"success","type":"bool"},{"internalType":"bytes","name":"returnData","type":"bytes"}],"internalType":"struct Multicall3.Result[]","name":"returnData","type":"tuple[]"}],"stateMutability":"payable","type":"function"}]')

In [50]:
# Contract Addresses
UNIV3_USDC_ETH='0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640'

In [51]:
# Function Signatures 4 bytes
getBlocknumber_4b = '42cbb15c'
getBloclTimestamp_4b= '0f28c97d'
getReserves_4b = '0902f1ac'
slot0_4b = '3850c7bd'

In [74]:
# Functions
def bytes_to_hexstr(b: any) -> str:
    if isinstance(b,list):
        return [bytes_to_hexstr(a) for a in b]
    return '0x' + b.hex()

def decode_outputdata_uniV3_price(b: bytes) -> list[float]:
    aggregated_data_uniV3 = decode(['(bool,bytes)[]'], b)[0]

    # UNI-V3
    # slot0():
    # sqrtPriceX96 uint160, tick int24, observationIndex uint16, observationCardinality uint16, observationCardinalityNext uint16, feeProtocol uint8, unlocked bool
    # 'uint160', 'int24', 'uint16', 'uint16', 'uint16', 'uint8', 'bool'
    usdc_eth_slot0_raw = aggregated_data_uniV3[0]
    usdc_eth_slot0_sqrt_ratioX96 = decode(['uint160', 'int24', 'uint16', 'uint16', 'uint16', 'uint8', 'bool'], usdc_eth_slot0_raw[1])[0]
    usdc_eth_price = usdc_eth_slot0_sqrt_ratioX96**2 / 2**192 /1e12
    eth_usdc_price_v3 = 1/usdc_eth_price

    # Timestamp
    timestamp_raw = aggregated_data_uniV3[-1]
    timestamp = int(timestamp_raw[1].hex(),16)

    block_raw = aggregated_data_uniV3[-2]
    block_number = int(block_raw[1].hex(),16)
    
    return [eth_usdc_price_v3, timestamp, block_number]
    

In [75]:
# web3 instance, function from web3py
w3 = web3.Web3()
m3 = w3.eth.contract(address = MULTICALL3_ADDRESS, abi=MULTICALL3_ABI)

In [76]:
# Arguments fro the tryAggregate Fuunction
aggregate_calldata = [
    [
        UNIV3_USDC_ETH,
        f'0x{slot0_4b}',
    ],
    [
        MULTICALL3_ADDRESS,
        f'0x{getBloclTimestamp_4b}',
    ],
    [
        MULTICALL3_ADDRESS,
        f'0x{getBlocknumber_4b}',
    ],
]

In [77]:
aggregate_calldata

[['0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640', '0x3850c7bd'],
 ['0xcA11bde05977b3631167028862bE2a173976CA11', '0x0f28c97d'],
 ['0xcA11bde05977b3631167028862bE2a173976CA11', '0x42cbb15c']]

In [78]:
# aggregate_calldata list of list
# Generate calldate (the input) via m3 Multicall3 encode ABIfor cryo - calldata is in Hex format
calldata = m3.encode_abi("tryAggregate", args=[False, aggregate_calldata])

In [79]:
calldata

'0xbce38bd7000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000400000000000000000000000000000000000000000000000000000000000000003000000000000000000000000000000000000000000000000000000000000006000000000000000000000000000000000000000000000000000000000000000e0000000000000000000000000000000000000000000000000000000000000016000000000000000000000000088e6a0c2ddd26feeb64f039a2c41296fcb3f5640000000000000000000000000000000000000000000000000000000000000004000000000000000000000000000000000000000000000000000000000000000043850c7bd00000000000000000000000000000000000000000000000000000000000000000000000000000000ca11bde05977b3631167028862be2a173976ca11000000000000000000000000000000000000000000000000000000000000004000000000000000000000000000000000000000000000000000000000000000040f28c97d00000000000000000000000000000000000000000000000000000000000000000000000000000000ca11bde05977b3631167028862be2a173976ca1100000000000000000000000000000

In [80]:
# cryo.collect() using calldata 
cryo_kwargs = {
    'rpc': 'https://eth.merkle.io',
    'blocks': ['-10:latest'], 
}
            
eth_call_uni_df = cryo.collect(
    'eth_calls',
    to_address = [MULTICALL3_ADDRESS],
    call_data=[calldata],
     output_format="polars",
    **cryo_kwargs,
)

In [81]:
output_data=eth_call_uni_df['output_data'][0]

In [82]:
output_data

b"\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00 \x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x03\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00`\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\xa0\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x02 \x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00@\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

In [83]:
# Function decode_outputdata_uniV3_price uses eth_abi.decode to get decimals values from binary format
prices = [decode_outputdata_uniV3_price(x) for x in eth_call_uni_df['output_data'].to_list()]

In [84]:
prices

[[2585.8227427035713, 22497474, 1747420895],
 [2585.8227427035713, 22497475, 1747420907],
 [2585.8227427035713, 22497476, 1747420919],
 [2584.653075674836, 22497477, 1747420931],
 [2584.653075674836, 22497478, 1747420943],
 [2584.653075674836, 22497479, 1747420955],
 [2584.653075674836, 22497480, 1747420967],
 [2584.64399653122, 22497481, 1747420979],
 [2584.64399653122, 22497482, 1747420991],
 [2584.64399653122, 22497483, 1747421003]]